In [1]:
import pipelines
import pandas as pd
from sklearn.linear_model import Lasso, Ridge, LinearRegression

df = pd.read_csv('hotel_bookings.csv')
df = df.drop_duplicates()

In [ ]:
from sklearn.pipeline import Pipeline
lasso_reg_pipe = Pipeline(
    [
        ('Data Preparation', pipelines.default_data_preparation_pipeline),
        ('model', Lasso())
    ]
)
lasso_reg_pipe

Pipeline(steps=[('Data Preparation',
                 Pipeline(steps=[('Feature Engineering', FeatureEngineer()),
                                 ('feature encoding',
                                  ColumnTransformer(transformers=[('categorial_oh',
                                                                   OneHotEncoder(handle_unknown='ignore'),
                                                                   ['hotel',
                                                                    'arrival_date_month',
                                                                    'meal',
                                                                    'market_segment',
                                                                    'distribution_channel',
                                                                    'reserved_room_type',
                                                                    'deposit_type',
                                                                    'customer_type',
                                                                    'country_group']),
                                                                  ('nu...
                                                                    'stays_in_week_nights',
                                                                    'adults',
                                                                    'children',
                                                                    'babies',
                                                                    'is_repeated_guest',
                                                                    'previous_cancellations',
                                                                    'previous_bookings_not_canceled',
                                                                    'booking_changes',
                                                                    'days_in_waiting_list',
                                                                    'required_car_parking_spaces',
                                                                    'total_of_special_requests',
                                                                    'total_nights',
                                                                    'is_weekend_stay',
                                                                    'total_guests',
                                                                    'has_children',
                                                                    'booking_intensity',
                                                                    'has_agent',
                                                                    'has_company'])]))])),
                ('model', Lasso())])

In [5]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 16.5 MB/s eta 0:00:00


## Lasso

In [ ]:
lasso_reg_pipe.get_params()

{'memory': None,
 'steps': [('Data Preparation',
   Pipeline(steps=[('Feature Engineering', FeatureEngineer()),
                   ('feature encoding',
                    ColumnTransformer(transformers=[('categorial_oh',
                                                     OneHotEncoder(handle_unknown='ignore'),
                                                     ['hotel',
                                                      'arrival_date_month', 'meal',
                                                      'market_segment',
                                                      'distribution_channel',
                                                      'reserved_room_type',
                                                      'deposit_type',
                                                      'customer_type',
                                                      'country_group']),
                                                    ('numeric',
                                  

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, train_test_split
import numpy as np
X = df.drop(columns='adr')
y = df.adr
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=22)
def objective(trial):
    tol = trial.suggest_float('model__tol', 1e-8, 1e-4, log=True)
    alpha = trial.suggest_float('model__alpha', 1e-3, 2, log=True)

    lasso_reg_pipe.set_params(
        model__tol=tol,
        model__alpha=alpha

    )

    cv = KFold(n_splits=5, shuffle=True, random_state=22)
    scores = cross_val_score(
        lasso_reg_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [ ]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=10)

[I 2026-06-16 21:28:05,651] A new study created in memory with name: no-name-00a35f4b-13fb-4a71-9508-b4f61482aa8c


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-06-16 21:28:22,753] Trial 0 finished with value: -23.10670470021514 and parameters: {'model__tol': 9.252827894985906e-06, 'model__alpha': 0.09520189671563649}. Best is trial 0 with value: -23.10670470021514.
[I 2026-06-16 21:28:26,326] Trial 1 finished with value: -25.895454625635658 and parameters: {'model__tol': 4.628644184615139e-05, 'model__alpha': 0.6897687857096326}. Best is trial 0 with value: -23.10670470021514.
[I 2026-06-16 21:28:53,568] Trial 2 finished with value: -23.011680270507913 and parameters: {'model__tol': 1.9948996567272196e-05, 'model__alpha': 0.035855272897909116}. Best is trial 2 with value: -23.011680270507913.
[I 2026-06-16 21:28:57,946] Trial 3 finished with value: -25.97848252327801 and parameters: {'model__tol': 6.234172956681686e-07, 'model__alpha': 0.7100366110711929}. Best is trial 2 with value: -23.011680270507913.
[I 2026-06-16 21:29:29,723] Trial 4 finished with value: -23.008521413840143 and parameters: {'model__tol': 2.5008109285999726e-06, 

In [ ]:
study.trials_dataframe().to_csv('lin_reg_l1.csv')

## Ridge

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, train_test_split
import numpy as np
X = df.drop(columns='adr')
y = df.adr
ridge_reg_pipe = Pipeline(
    [
        ('Data Preparation', pipelines.default_data_preparation_pipeline),
        ('model', Ridge())
    ]
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=22)
def objective(trial):
    tol = trial.suggest_float('model__tol', 1e-8, 1e-4, log=True)
    alpha = trial.suggest_float('model__alpha', 1e-3, 2, log=True)

    ridge_reg_pipe.set_params(
        model__tol=tol,
        model__alpha=alpha

    )

    cv = KFold(n_splits=5, shuffle=True, random_state=22)
    scores = cross_val_score(
        ridge_reg_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=10)

[I 2026-06-16 21:34:22,401] A new study created in memory with name: no-name-2ba9ba24-40e8-40b8-a132-1280559ae803


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-06-16 21:34:25,511] Trial 0 finished with value: -22.991464062751295 and parameters: {'model__tol': 6.313563292411064e-07, 'model__alpha': 0.047080612664201445}. Best is trial 0 with value: -22.991464062751295.
[I 2026-06-16 21:34:28,120] Trial 1 finished with value: -22.991561802800742 and parameters: {'model__tol': 6.321009444424724e-08, 'model__alpha': 0.07337957475176125}. Best is trial 0 with value: -22.991464062751295.
[I 2026-06-16 21:34:30,341] Trial 2 finished with value: -22.99204502946852 and parameters: {'model__tol': 1.8977464633846232e-06, 'model__alpha': 0.2504754434804079}. Best is trial 0 with value: -22.991464062751295.
[I 2026-06-16 21:34:32,647] Trial 3 finished with value: -22.991775562068987 and parameters: {'model__tol': 2.8972422725296425e-06, 'model__alpha': 0.15035060579557882}. Best is trial 0 with value: -22.991464062751295.
[I 2026-06-16 21:34:34,926] Trial 4 finished with value: -22.99152485534857 and parameters: {'model__tol': 3.4203282448551646e-

In [ ]:
study.trials_dataframe().to_csv('lin_reg_l2.csv')

# Random Forest

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split

rf_pipe = Pipeline(

    [('preparation', pipelines.default_data_preparation_pipeline),
     ('model', RandomForestRegressor())]
)
df = df[~df.adr.isna()]

X = df.drop(columns='adr')
y = df.adr
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=22)
def objective_rf(trial):
    # Гиперпараметры RandomForest
    n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
    max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)
    min_samples_split = trial.suggest_int('model__min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('model__min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('model__max_features', ['sqrt', 'log2', None])
    bootstrap = trial.suggest_categorical('model__bootstrap', [True, False])

    # Устанавливаем параметры в пайплайн
    rf_pipe.set_params(
        model__n_estimators=n_estimators,
        model__max_depth=max_depth,
        model__min_samples_split=min_samples_split,
        model__min_samples_leaf=min_samples_leaf,
        model__max_features=max_features,
        model__bootstrap=bootstrap
    )

    # Кросс-валидация
    cv = KFold(n_splits=3, shuffle=True, random_state=22)
    scores = cross_val_score(
        rf_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error',          # метрика для классификации
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [15]:
import optuna
import numpy as np
study = optuna.create_study(direction='maximize')
study.optimize(objective_rf, show_progress_bar=True, n_trials=10, n_jobs=-1)

[I 2026-06-16 22:33:30,336] A new study created in memory with name: no-name-d55d77c7-3a7a-44d6-bb8c-7d656704983e


  0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_11918/119032077.py:17: UserWarning: The distribution is specified by [50, 500] and step=60, but the range is not divisible by `step`. It will be replaced with [50, 470].
  n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
/tmp/ipykernel_11918/119032077.py:18: UserWarning: The distribution is specified by [3, 20] and step=5, but the range is not divisible by `step`. It will be replaced with [3, 18].
  max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)


[I 2026-06-16 22:34:06,166] Trial 1 finished with value: -22.43711164043655 and parameters: {'model__n_estimators': 470, 'model__max_depth': 8, 'model__min_samples_split': 9, 'model__min_samples_leaf': 17, 'model__max_features': 'sqrt', 'model__bootstrap': True}. Best is trial 1 with value: -22.43711164043655.
[I 2026-06-16 22:34:06,829] Trial 0 finished with value: -22.55805999476109 and parameters: {'model__n_estimators': 230, 'model__max_depth': 8, 'model__min_samples_split': 8, 'model__min_samples_leaf': 7, 'model__max_features': 'log2', 'model__bootstrap': False}. Best is trial 1 with value: -22.43711164043655.
[I 2026-06-16 22:35:18,780] Trial 3 finished with value: -16.53549341422412 and parameters: {'model__n_estimators': 170, 'model__max_depth': 18, 'model__min_samples_split': 4, 'model__min_samples_leaf': 16, 'model__max_features': 'sqrt', 'model__bootstrap': False}. Best is trial 3 with value: -16.53549341422412.
[I 2026-06-16 22:35:41,739] Trial 2 finished with value: -14.1

In [16]:
df.adr.std()

60.07365081761906

In [17]:
study.trials_dataframe().to_csv('rf_reg.csv')